# 読む前に、AIがファイルを選ぶ：Jev × DuckDB × GeoParquet を手で確かめる

このノートブックでは、#30DayMapChallenge 2026 Day 27 と FOSS4G Hokkaido 2026 の発表で使った処理を、1 段ずつ動かして確かめます。

1. GeoParquet と目録（manifest）を見る
2. DuckDB の SQL から Jev を呼ぶ
3. jev 拡張の 4 つの仕組み（20 行まとめて送る・同時 16 リクエスト・キャッシュ・使用量の集計）を確かめる
4. 目録を Jev に読ませて、読むファイルを選ぶ
5. 全ファイル（方式A）と、Jev がファイルを選ぶ方式（方式C）を比べる
6. 結果を地図で見る

**費用の目安：** 北広島市（589 件）だけを使います。最後まで実行して、入力トークンは 20〜30 万程度です。jev 拡張の推定では $0.01 前後です。

**必要なもの：** TypeSafe の API キー（https://console.typesafe.ai/keys ）

データ：OpenStreetMap（© OpenStreetMap contributors, ODbL）。判定は地物の属性だけに基づくもので、実際の安全性や営業状況を保証しません。

## 0. 準備

### API キーを Colab のシークレットに登録する

ノートブックの本文にキーを書かないでください。

1. 左側の鍵のアイコン（シークレット）を開きます。
2. 名前を `TYPESAFE_API_KEY` にして、値にキーを入れます。
3. 「ノートブックからのアクセス」をオンにします。

jev 拡張は、環境変数 `TYPESAFE_API_KEY` からキーを読みます。

In [ ]:
!pip install -q duckdb==1.5.5 folium
!git clone -q --depth 1 https://github.com/shinyanakashima/30daymapchallenge-2026day27-jev-geoparquet.git repo 2>/dev/null || echo "already cloned"

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
except ImportError:
    pass  # Colab の外では、環境変数 TYPESAFE_API_KEY を設定してから起動してください
print("API key set:", bool(os.environ.get("TYPESAFE_API_KEY")))

In [ ]:
import json, time
import duckdb

DATA = "repo/data/kitahiroshima"
ATTR_COLS = ["name", "name_en", "amenity", "leisure", "shop", "tourism", "historic", "office",
             "healthcare", "craft", "sport", "club", "cuisine", "operator", "brand", "access", "fee",
             "opening_hours", "description", "religion", "building", "playground", "indoor"]
# 値のある属性だけを "key: value; key: value" の 1 つの文字列にする（ジオメトリは送らない）
ATTRS = "concat_ws('; ', " + ", ".join(f"'{c}: ' || {c}" for c in ATTR_COLS) + ")"
OPTIONS = ['supported', 'contradicted', 'insufficient_information']

def connect():
    con = duckdb.connect()
    con.sql("INSTALL spatial; LOAD spatial; INSTALL jev FROM community; LOAD jev;")
    if os.environ.get("JEV_API_URL"):  # 中継やモックを使う場合だけ
        con.sql(f"SET jev_api_url = '{os.environ['JEV_API_URL']}'")
    return con

def stats(con):
    return json.loads(con.sql("SELECT jev_stats()").fetchone()[0])

def diff(a, b):
    keys = ["requests", "input_tokens", "rows_evaluated", "cache_hits", "estimated_cost_usd"]
    return {k: round(b[k] - a[k], 6) for k in keys}

def q_feature(q):
    return f'Does this place match the request "{q}"? Judge only from the attributes given.'

con = connect()
print(con.sql("SELECT jev_version()").fetchone()[0])

## 1. GeoParquet と目録を見る

北広島市の地物 589 件を、カテゴリ（5 種）× 区域（4 つ）で 20 個の GeoParquet に分けてあります。

In [ ]:
con.sql(f"""
SELECT regexp_extract(filename, '([^/]+)\\.parquet$', 1) AS file, count(*) AS rows
FROM read_parquet('{DATA}/parts/*.parquet', filename = true)
GROUP BY ALL ORDER BY file
""").show(max_rows=25)

目録（`manifest.json`）には、ファイルごとの要約が入っています。Jev が読むのは、この要約だけです。

In [ ]:
manifest = json.load(open(f"{DATA}/manifest.json"))
entry = manifest["files"][0]
print(json.dumps(entry, ensure_ascii=False, indent=1))

## 2. SQL から Jev を呼ぶ

`jev_choice(状態, 質問, 選択肢)` は、選択肢の中から最も確からしいものを返します。ここでは公園のファイルから 8 件だけ判定します。

In [ ]:
QUESTION = "A place where children can play"
before = stats(con)
con.sql(f"""
SELECT name, leisure,
       {ATTRS} AS sent_to_jev,
       jev_choice(struct_pack(attributes := {ATTRS}), '{q_feature(QUESTION)}', {OPTIONS}) AS judgment
FROM read_parquet('{DATA}/parts/parks_play__*.parquet')
LIMIT 8
""").show(max_width=200)
print(diff(before, stats(con)))

## 3. jev 拡張の 4 つの仕組みを確かめる

### 3-1. 20 行をまとめて 1 リクエスト（`jev_batch_size`）

同じ 60 行を、1 行ずつ送る場合と 20 行ずつ送る場合で比べます。リクエスト数とトークンを見てください。

まとめて送ると、リクエスト数とトークンは減ります。一方で、一緒に送る行によって答えが変わることもあります。最後に出る「same answer」で、一致した行の数を確かめてください。数値は実行のたびに少し変わります。

In [ ]:
def judge_rows(con, sql_from, question, label):
    con.sql("SELECT jev_cache_clear()").fetchall()
    s0, t0 = stats(con), time.perf_counter()
    rows = con.sql(f"""SELECT osm_id, jev_choice(struct_pack(attributes := {ATTRS}),
                       '{q_feature(question)}', {OPTIONS}) FROM {sql_from}""").fetchall()
    t = time.perf_counter() - t0
    d = diff(s0, stats(con))
    print(f"{label:28s} rows={len(rows):4d} requests={d['requests']:4d} input_tokens={d['input_tokens']:7d} time={t:.2f}s")
    return dict(rows)

SAMPLE = f"(SELECT * FROM read_parquet('{DATA}/parts/*.parquet') ORDER BY osm_id LIMIT 60)"
con.sql("SET jev_batch_size = 1")
r1 = judge_rows(con, SAMPLE, QUESTION, "batch_size = 1")
con.sql("SET jev_batch_size = 20")
r20 = judge_rows(con, SAMPLE, QUESTION, "batch_size = 20 (default)")
print("same answer:", sum(r1[k] == r20[k] for k in r1), "/", len(r1))

### 3-2. 同時 16 リクエスト（`jev_concurrency`）

全 589 件を、同時 1 リクエストと同時 16 リクエストで判定します。

In [ ]:
ALL = f"read_parquet('{DATA}/parts/*.parquet')"
con.sql("SET jev_concurrency = 1")
judge_rows(con, ALL, QUESTION, "concurrency = 1")
con.sql("SET jev_concurrency = 16")
_ = judge_rows(con, ALL, QUESTION, "concurrency = 16 (default)")

### 3-3. 同じ入力の答えはキャッシュされる

同じ問い合わせを、キャッシュを消さずにもう一度実行します。2 回目は API を呼ばずに答えが返ります（`cache_hits` が増え、トークンは増えません）。

In [ ]:
q = f"""SELECT count(*) FILTER (WHERE jev_choice(struct_pack(attributes := {ATTRS}),
         '{q_feature(QUESTION)}', {OPTIONS}) = 'supported') FROM {ALL}"""
con.sql("SELECT jev_cache_clear()").fetchall()
for i in (1, 2):
    s0, t0 = stats(con), time.perf_counter()
    n = con.sql(q).fetchone()[0]
    print(f"run {i}: supported={n} time={time.perf_counter() - t0:.2f}s", diff(s0, stats(con)))

### 3-4. 使用量を SQL で集計する（`jev_stats()`）

ここまでの合計です。発表の数字は、すべてこの値の差分から取りました。

589 行に対して `rows_evaluated` が少ないのは、属性の文字列がまったく同じ行（名前の無い公園など）を 1 回だけ評価しているためです。

In [ ]:
print(json.dumps(stats(con), indent=1))
print("distinct attribute strings:", con.sql(f"SELECT count(DISTINCT {ATTRS}) FROM {ALL}").fetchone()[0], "/ 589 rows")

## 4. 目録を Jev に読ませて、読むファイルを選ぶ

目録の各項目を「candidate（候補）／exclude（除外）／insufficient_information（判断材料不足）」に分けます。判断材料不足は、取りこぼしを避けるために読む側に残します。

In [ ]:
def file_question(q):
    return (f'This is a summary of one data file of places. Could the file contain places that match the '
            f'request "{q}"? Answer candidate if it may contain such places, exclude if it clearly does '
            f'not, insufficient_information if the summary is not enough to tell.')

def select_files(con, question):
    rows = [(f["id"], f["category_label"], f["rows"], f["unnamed_rows"],
             ", ".join(f"{k}: {v}" for k, v in f["tag_counts"].items()), ", ".join(f["sample_names"]))
            for f in manifest["files"]]
    con.sql("CREATE OR REPLACE TABLE manifest (id VARCHAR, category VARCHAR, rows INTEGER, unnamed_rows INTEGER, "
            "most_common_tags VARCHAR, sample_names VARCHAR)")
    con.executemany("INSERT INTO manifest VALUES (?, ?, ?, ?, ?, ?)", rows)
    q = file_question(question).replace("'", "''")
    return dict(con.sql(f"""SELECT id, jev_choice(struct_pack(category, rows, unnamed_rows, most_common_tags,
                             sample_names), '{q}', ['candidate', 'exclude', 'insufficient_information'])
                             FROM manifest""").fetchall())

QUESTION = "A place to buy food or daily necessities"
con.sql("SELECT jev_cache_clear()").fetchall()
status = select_files(con, QUESTION)
for fid, s in sorted(status.items()):
    print(f"{fid:24s} {s}")

## 5. 方式A（全ファイル）と方式C（Jev が選ぶ）を比べる

空間条件（市の範囲）・SQL・地物の判定は同じです。違うのは、読むファイルだけです。

候補保持率は、方式Aで supported になった地物のうち、方式Cが読んだファイルに含まれていたものの割合です。

In [ ]:
boundary = json.load(open(f"{DATA}/boundary.geojson"))
WKT = con.sql(f"SELECT ST_AsText(ST_GeomFromGeoJSON('{json.dumps(boundary['geometry'])}'))").fetchone()[0]
by_id = {f["id"]: f for f in manifest["files"]}

def run(con, question, method):
    con.sql("SELECT jev_cache_clear()").fetchall()
    s0, t0 = stats(con), time.perf_counter()
    st = select_files(con, question) if method == "C" else {fid: "read" for fid in by_id}
    t1 = time.perf_counter()
    paths = [f"{DATA}/{by_id[f]['file']}" for f, s in st.items() if s != "exclude"]
    rows = con.sql(f"""SELECT osm_id, name, ST_X(geometry) lon, ST_Y(geometry) lat, regexp_extract(filename, '([^/]+)\\.parquet$', 1) AS file,
                       jev_choice(struct_pack(attributes := {ATTRS}), '{q_feature(question)}', {OPTIONS}) AS j
                       FROM read_parquet({paths}, filename = true)
                       WHERE ST_Within(geometry, ST_GeomFromText('{WKT}'))""").fetchall()
    t2 = time.perf_counter()
    d = diff(s0, stats(con))
    return {"status": st, "rows": rows, "select_s": t1 - t0, "total_s": t2 - t0, **d}

res = {m: run(con, QUESTION, m) for m in ("A", "C")}
ref = {r[0] for r in res["A"]["rows"] if r[5] == "supported"}
read_c = {f for f, s in res["C"]["status"].items() if s != "exclude"}
kept = {r[0] for r in res["A"]["rows"] if r[0] in ref and r[4] in read_c}
for m, r in res.items():
    print(f"method {m}: files={sum(s != 'exclude' for s in r['status'].values()):2d} rows={len(r['rows']):3d} "
          f"total={r['total_s']:.2f}s (select {r['select_s']:.2f}s) input_tokens={r['input_tokens']}")
print(f"retention (C): {len(kept)}/{len(ref)}")

1 回だけの実行では、最初の呼び出しの遅れなどで、時間の順位が入れ替わることがあります。

発表の計測では、各方式を 3 回ずつ実行し、中央値で比べました。

- 北広島市（589 件）では、方式Cの方が遅くなりました。判定は同時リクエスト 1〜2 往復で終わるため、ファイル選択の約 0.25 秒を取り戻せません。
- 札幌市（16,610 件）では、ファイルを除外できた質問で、方式Cの方が速くなりました。

## 6. 地図で見る

方式Cの結果です。破線の四角は、各ファイルの受け持ち範囲（データの bbox）です。地物がある位置そのものではありません。

In [ ]:
import folium
COLORS = {"candidate": "#0f7b5f", "insufficient_information": "#d49a12", "exclude": "#7d8894"}
m = folium.Map(location=[42.97, 141.56], zoom_start=12, tiles="OpenStreetMap")
folium.GeoJson(boundary, style_function=lambda f: {"color": "#333", "weight": 1.5, "fill": False}).add_to(m)
for fid, s in res["C"]["status"].items():
    x0, y0, x1, y1 = by_id[fid]["bbox"]
    folium.Rectangle([[y0, x0], [y1, x1]], color=COLORS[s], weight=1.5, dash_array="6 4", fill=False,
                     tooltip=f"{fid}: {s}").add_to(m)
for osm_id, name, lon, lat, f, j in res["C"]["rows"]:
    if j == "supported":
        folium.CircleMarker([lat, lon], radius=5, color="#fff", weight=1, fill=True, fill_color="#0f7b5f",
                            fill_opacity=1, tooltip=f"{name or '(no name)'} · {osm_id}").add_to(m)
m

## 7. 自分の質問で試す

質問を書き換えて、方式Cを実行してみてください（英語の方が安定します）。

In [ ]:
MY_QUESTION = "A place to eat ramen"
r = run(con, MY_QUESTION, "C")
print({k: r[k] for k in ("total_s", "select_s", "requests", "input_tokens", "estimated_cost_usd")})
print([x[1] for x in r["rows"] if x[5] == "supported"][:20])

## 次へ

- 札幌市で試す場合は、`DATA = "repo/data/sapporo"` に変えて、`connect()` からやり直します。全件の判定 1 回で入力トークンが約 145 万（jev 拡張の推定で約 $0.06）かかります。
- 同じ処理を API として動かす手順は、`tutorial/README.md`（Docker）と `container/README.md`（Cloudflare Containers + R2）にあります。